## Camera Pose Estimation for Tennis Match - UE Synthetic Data

This notebook documents a reproducible pipeline for estimating camera pose in tennis match footage using court-line segmentation. We leverage synthetically annotated data generated in Unreal Engine to avoid manual labeling, significantly reducing human annotation effort.

The notebook focuses on the deep-learning components of the workflow: model architecture, training, and inference. We also illustrate how the trained model can be applied to a real-world sports scenario — a recent Australian Open match between Alcaraz and Paul.

<p align="center">
    <img src="../rsc/render1frame0016.png" alt="synthetic_image" height = "350" width="600">
    <img src="../rsc/inference_example.png" alt="real_infernece_image" height = "350" width="600">
</p>

## Roadmap

1. Model: design and implementation of the CNN segmentation architecture
2. Training: dataset preparation (UE synthetic data), training pipeline using PyTorch, and evaluation metrics
3. Inference: segmentation-to-pose pipeline, post-processing algorithms, and application to a real match

## 1. U-Net - Deep Learning architecture for Image Segmentation
<p align="center">
    <img src="../rsc/u-net-architecture.png" alt="U-Net" height = "350" width="600">
    <img src="../rsc/road-Unet.jpg" alt="U-Net" height = "350" width="600">
</p>


U-Net is a convolutional neural network architecture designed for image segmentation, especially in biomedical applications. It follows an encoder–decoder structure: the contracting path (encoder) captures context through successive convolution and pooling layers, while the expansive path (decoder) enables precise localization by upsampling feature maps. A key feature of U-Net is the use of skip connections that link corresponding layers in the encoder and decoder, allowing the model to combine low-level spatial information with high-level semantic features. This design makes U-Net highly effective for tasks requiring accurate pixel-level predictions, even with limited training data.



In [2]:
"""The first thing we want is to load the synthetic tennis dataset with the image
and the corresponding mask from our demo dataset for this notebook"""

import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np


class TennisDemoDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_transform = None):
        super().__init__()
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_transform = img_transform
        self.img_files = sorted(os.listdir(img_dir))
        self.mask_files = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.img_files)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        if self.img_transform:
            image = self.img_transform(image)
        mask = torch.from_numpy(np.array(mask)).long() # we dont want to use image transform if there is resize cause maybe destroys the interpolation between pixels in the mask
        return image, mask



In [6]:
"""The next step that we want to do is to create the layers of the model.
From the encoder, to the botteleneck, to the decoder. We will use the classic UNet architecture as we said."""

import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down = nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.down(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.double_conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x2, x1], dim=1)
        return self.double_conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

In [7]:
"""With the previous layers defined, we can now define the UNet architecture."""

class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)
        self.outc = OutConv(64, num_classes)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)


## 2. Training Pipeline for U-Net on a Custom Dataset

This section describes the end-to-end training pipeline used to train the U-Net architecture from scratch on a custom demo dataset(30 images and 30 masks). It covers the key steps involved, including data loading and preprocessing, dataset splitting, and the application of data augmentation techniques to improve generalization. The section also outlines the model configuration, loss function, and optimization strategy, followed by the implementation of the training and validation loops. Finally, it presents the evaluation metrics used to assess performance, including training and validation loss as well as Intersection over Union (IoU) to measure pixel-wise prediction accuracy.

<p align = "center">
    <img src="../rsc/CNN_Training_Loop.jpg" alt="forward_backward propagation" height = "400" width="700">
</p>

